# Sleep Stage Classification — YDL 2026

## Шаг 1: Загрузка и первичный анализ данных

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sub   = pd.read_csv('sample_submission.csv')

print('train:', train.shape)
print('test: ', test.shape)
print('sub:  ', sub.shape)

train: (9000, 23)
test:  (5000, 22)
sub:   (5000, 2)


In [2]:
print('=== dtypes ===')
print(train.dtypes)
print('\n=== Missing values (train) ===')
print(train.isnull().sum()[train.isnull().sum() > 0])

=== dtypes ===
id                           int64
eeg_delta_power            float64
eeg_theta_power            float64
eeg_alpha_power            float64
eeg_sigma_power            float64
eeg_beta_power             float64
eeg_gamma_power            float64
eeg_slow_osc_power         float64
eeg_spectral_entropy       float64
eeg_spindle_density        float64
eeg_kcomplex_rate          float64
emg_chin_tone              float64
emg_tone_variance          float64
eog_movement_density       float64
eog_amplitude              float64
heart_rate_mean            float64
heart_rate_variability     float64
respiration_rate           float64
respiration_variability    float64
spo2_mean                  float64
body_movement_index        float64
eog_burst_index            float64
sleep_stage                  int64
dtype: object

=== Missing values (train) ===
eog_burst_index    4501
dtype: int64


In [3]:
print('=== Распределение классов (value_counts) ===')
vc = train['sleep_stage'].value_counts().sort_index()
print(vc)
print('\nДоля каждого класса:')
print((vc / len(train)).round(3))

=== Распределение классов (value_counts) ===
sleep_stage
0    2001
1    2442
2    2237
3    2320
Name: count, dtype: int64

Доля каждого класса:
sleep_stage
0    0.222
1    0.271
2    0.249
3    0.258
Name: count, dtype: float64


In [5]:
train.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
id,9000.0,4499.500,2598.221,0.000,2249.750,4499.500,6749.250,8999.000
eeg_delta_power,9000.0,-0.015,2.431,-9.567,-1.688,-0.012,1.622,8.798
eeg_theta_power,9000.0,-0.034,2.408,-8.277,-1.649,-0.020,1.604,9.222
eeg_alpha_power,9000.0,-0.016,5.042,-20.632,-3.465,0.019,3.419,18.870
eeg_sigma_power,9000.0,0.012,2.207,-7.948,-1.460,-0.001,1.512,9.603
eeg_beta_power,9000.0,-0.010,2.883,-11.460,-1.963,0.005,1.956,10.027
eeg_gamma_power,9000.0,-0.011,2.428,-9.638,-1.652,0.010,1.635,8.296
eeg_slow_osc_power,9000.0,-0.014,2.801,-14.123,-1.873,0.023,1.866,10.362
eeg_spectral_entropy,9000.0,0.010,2.753,-9.301,-1.859,-0.028,1.882,10.160
eeg_spindle_density,9000.0,0.006,2.174,-8.828,-1.484,0.024,1.479,8.937


In [4]:
# eog_burst_index отдельно — пропуски в обоих сплитах
print('eog_burst_index missing — train:', train['eog_burst_index'].isnull().sum(),
      '/', len(train),
      f"({train['eog_burst_index'].isnull().mean():.1%})")
print('eog_burst_index missing — test: ', test['eog_burst_index'].isnull().sum(),
      '/', len(test),
      f"({test['eog_burst_index'].isnull().mean():.1%})")


eog_burst_index missing — train: 4501 / 9000 (50.0%)
eog_burst_index missing — test:  2477 / 5000 (49.5%)


---
**Итог шага 1:**
- Train: 9000 строк, 22 колонки (id + 20 признаков + таргет). Test: 5000 строк.
- Все признаки — float64, пропуски только у `eog_burst_index` (~50% в обоих сплитах).
- 4 класса сна (0–3); баланс классов нужно проверить — возможен дисбаланс.
- Значения признаков стандартизированы (диапазон примерно –5 … +5) — шкалирование, вероятно, уже применено.
---

## Шаг 2: Обработка пропусков в eog_burst_index

In [5]:
# Медиана считается только по train — чтобы не допускать утечки данных из test
eog_median = train['eog_burst_index'].median()
print(f'Медиана eog_burst_index (train): {eog_median:.4f}')

for df in [train, test]:
    df['eog_burst_missing'] = df['eog_burst_index'].isnull().astype(int)
    df['eog_burst_index']   = df['eog_burst_index'].fillna(eog_median)

print('\nПропуски после заполнения:')
print('train:', train['eog_burst_index'].isnull().sum(),
      '| flag sum:', train['eog_burst_missing'].sum())
print('test: ', test['eog_burst_index'].isnull().sum(),
      '| flag sum:', test['eog_burst_missing'].sum())

Медиана eog_burst_index (train): -0.0022

Пропуски после заполнения:
train: 0 | flag sum: 4501
test:  0 | flag sum: 2477


In [6]:
# Проверка: новая колонка появилась, пропусков нет
print('Новые колонки:', [c for c in train.columns if 'eog_burst' in c])
train[['eog_burst_index', 'eog_burst_missing']].describe().T.round(4)

Новые колонки: ['eog_burst_index', 'eog_burst_missing']


,count,mean,std,min,25%,50%,75%,max
eog_burst_index,9000.0,-0.0062,0.7338,-3.9363,-0.0022,-0.0022,-0.0022,3.9359
eog_burst_missing,9000.0,0.5001,0.5000,0.0000,0.0000,1.0000,1.0000,1.0000


---
**Итог шага 2:**
- Медиана вычислена **только по train** — утечки из test нет.
- `eog_burst_index`: пропуски заполнены медианой в обоих сплитах.
- `eog_burst_missing`: бинарный флаг (1 = было пусто), позволяет модели учитывать сам факт отсутствия значения.
---

## Эксперименты: сравнение моделей и стратегий признаков

In [7]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Полный набор признаков (22): медиана-заполнение + флаг уже применены в шаге 2
FEATURES_ALL = [c for c in train.columns if c not in ('id', 'sleep_stage')]
# Без eog_burst_index и флага (эксперимент 1)
FEATURES_NO_EOG = [c for c in FEATURES_ALL if c not in ('eog_burst_index', 'eog_burst_missing')]

X_all    = train[FEATURES_ALL].values
X_no_eog = train[FEATURES_NO_EOG].values
y        = train['sleep_stage'].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'FEATURES_ALL    : {len(FEATURES_ALL)} признаков')
print(f'FEATURES_NO_EOG : {len(FEATURES_NO_EOG)} признаков')

BASELINE_F1 = 0.7963  # RF n=300, все 22 признака

FEATURES_ALL    : 22 признаков
FEATURES_NO_EOG : 20 признаков


In [8]:
# --- Эксперимент 1: RF без eog_burst_index и eog_burst_missing ---
scores_exp1 = cross_val_score(
    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    X_no_eog, y, cv=cv, scoring='f1_macro'
)
print('=' * 55)
print('Эксперимент 1 — RF, без eog_burst_index (20 признаков)')
print('=' * 55)
print(f'  Folds : {scores_exp1.round(4)}')
print(f'  Mean  : {scores_exp1.mean():.4f}')
print(f'  Std   : {scores_exp1.std():.4f}')
print(f'  vs Baseline ({BASELINE_F1}): {scores_exp1.mean() - BASELINE_F1:+.4f}')

Эксперимент 1 — RF, без eog_burst_index (20 признаков)
  Folds : [0.7675 0.7637 0.781  0.7758 0.7756]
  Mean  : 0.7727
  Std   : 0.0062
  vs Baseline (0.7963): -0.0236


In [9]:
# --- Эксперимент 2: ExtraTreesClassifier (22 признака) ---
scores_exp2 = cross_val_score(
    ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    X_all, y, cv=cv, scoring='f1_macro'
)
print('=' * 55)
print('Эксперимент 2 — ExtraTrees (22 признака)')
print('=' * 55)
print(f'  Folds : {scores_exp2.round(4)}')
print(f'  Mean  : {scores_exp2.mean():.4f}')
print(f'  Std   : {scores_exp2.std():.4f}')
print(f'  vs Baseline ({BASELINE_F1}): {scores_exp2.mean() - BASELINE_F1:+.4f}')

Эксперимент 2 — ExtraTrees (22 признака)
  Folds : [0.7944 0.7956 0.8001 0.8063 0.7994]
  Mean  : 0.7992
  Std   : 0.0042
  vs Baseline (0.7963): +0.0029


In [10]:
# --- Эксперимент 3: GradientBoostingClassifier (22 признака) ---
scores_exp3 = cross_val_score(
    GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
    X_all, y, cv=cv, scoring='f1_macro'
)
print('=' * 55)
print('Эксперимент 3 — GradientBoosting (22 признака)')
print('=' * 55)
print(f'  Folds : {scores_exp3.round(4)}')
print(f'  Mean  : {scores_exp3.mean():.4f}')
print(f'  Std   : {scores_exp3.std():.4f}')
print(f'  vs Baseline ({BASELINE_F1}): {scores_exp3.mean() - BASELINE_F1:+.4f}')

Эксперимент 3 — GradientBoosting (22 признака)
  Folds : [0.8082 0.8009 0.8138 0.8135 0.8203]
  Mean  : 0.8114
  Std   : 0.0065
  vs Baseline (0.7963): +0.0151


In [11]:
import pandas as pd

rows = [
    ('Baseline — RF n=300, 22 feat',          BASELINE_F1,              None),
    ('Exp 1 — RF n=300, без eog (20 feat)',   scores_exp1.mean(), scores_exp1.std()),
    ('Exp 2 — ExtraTrees n=300, 22 feat',     scores_exp2.mean(), scores_exp2.std()),
    ('Exp 3 — GradientBoosting, 22 feat',     scores_exp3.mean(), scores_exp3.std()),
]

summary = pd.DataFrame(rows, columns=['Model', 'Mean F1-macro', 'Std'])
summary['Mean F1-macro'] = summary['Mean F1-macro'].round(4)
summary['Std']           = summary['Std'].round(4)
summary['vs Baseline']   = (summary['Mean F1-macro'] - BASELINE_F1).round(4).apply(
                                lambda x: f'{x:+.4f}' if x != 0.0 else '—')
summary = summary.sort_values('Mean F1-macro', ascending=False).reset_index(drop=True)
summary.index += 1

print('=' * 70)
print('ИТОГОВАЯ ТАБЛИЦА (sorted by Mean F1-macro)')
print('=' * 70)
print(summary.to_string())
print(f'\nЛучшая модель: {summary.iloc[0]["Model"]}')

ИТОГОВАЯ ТАБЛИЦА (sorted by Mean F1-macro)
                                 Model  Mean F1-macro     Std vs Baseline
1    Exp 3 — GradientBoosting, 22 feat         0.8114  0.0065     +0.0151
2    Exp 2 — ExtraTrees n=300, 22 feat         0.7992  0.0042     +0.0029
3         Baseline — RF n=300, 22 feat         0.7963     NaN           —
4  Exp 1 — RF n=300, без eog (20 feat)         0.7727  0.0062     -0.0236

Лучшая модель: Exp 3 — GradientBoosting, 22 feat
